# SPARC Example 06: Peak Rotation Velocity Distribution

**EPS Research RAG Astrophysics Corpus — Unified HI Corpus v7.3**

Peak rotation velocity Vrot_max traces total galaxy mass.
This stacked histogram spans the full dynamic range —
from ~12 km/s dwarf irregulars to ~383 km/s massive spirals.

**Important note on corpus fidelity:** The `rotation_curve_corpus_v7.1_flat.csv` and `rotation_curve_corpus_v7.json` are **full-fidelity** — not a summary or veneer. The CSV contains every kinematic parameter published by Lelli et al. (2016) including per-galaxy inclination, distance uncertainties, mass-to-light ratios, and rotation curve statistics. The JSON adds full per-ring data: Vobs, Vgas, Vdisk, Vbul, errV at every radial point. This is the complete published dataset in a single machine-readable file.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.23442703  
**Source:** Lelli, McGaugh & Schombert (2016), AJ 152, 157  
**Dependencies:** Python 3, numpy, matplotlib, csv (standard library only)

In [ ]:
# ── Colab setup: auto-download corpus from Zenodo ─────────────
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import urllib.request
    CORPORA = {
        'rotation_curve_corpus_v7.json': 'https://zenodo.org/records/23442703/files/rotation_curve_corpus_v7.json',
    }
    for filename, url in CORPORA.items():
        if not os.path.exists(filename):
            print(f"Downloading {filename}...")
            urllib.request.urlretrieve(url, filename)
            print(f"  ✓ {filename}")
        else:
            print(f"  Already present: {filename}")
    print("Ready.")
else:
    print("Running locally — corpus files loaded from working directory.")


In [ ]:
import csv
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

rows = []
with open('rotation_curve_corpus_v7.1_flat.csv') as f:
    for r in csv.DictReader(f):
        try:
            rows.append({'survey': r['survey'],
                         'vmax': float(r['vrot_max_kms']),
                         'tier': int(r['quality_tier'])})
        except (ValueError, KeyError):
            pass

SURVEYS = ['SPARC', 'THINGS', 'LITTLE_THINGS', 'WALLABY']
COLORS  = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
bins = np.arange(0, 401, 20)  # through 400: peak velocities reach 383 km/s

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Panel 1: stacked histogram
ax = axes[0]
data = [[r['vmax'] for r in rows if r['survey']==s] for s in SURVEYS]
ax.hist(data, bins=bins, stacked=True, color=COLORS,
        label=[f'{s} (N={len(d)})' for s, d in zip(SURVEYS, data)],
        edgecolor='white', linewidth=0.5)
ax.set_xlabel('Peak Rotation Velocity Vmax (km/s)', fontsize=12)
ax.set_ylabel('N galaxies', fontsize=12)
ax.set_title('Vmax Distribution by Survey\nUnified HI Corpus v7.3', fontsize=11)
ax.legend(fontsize=9)

# Panel 2: CDF by survey
ax2 = axes[1]
for s, color in zip(SURVEYS, COLORS):
    v = sorted(r['vmax'] for r in rows if r['survey']==s)
    if v:
        ax2.plot(v, np.linspace(0, 1, len(v)), color=color, lw=2, label=s)
ax2.set_xlabel('Vmax (km/s)', fontsize=12)
ax2.set_ylabel('Cumulative fraction', fontsize=12)
ax2.set_title('Vmax CDF by Survey', fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.suptitle(f'HI Corpus v7.3 — Peak Velocity Distribution (N={len(rows)} galaxies)',
             fontsize=11)
plt.tight_layout()
plt.savefig('ex06_vrot_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

for s in SURVEYS:
    v = [r['vmax'] for r in rows if r['survey']==s]
    if v:
        print(f'{s:15s}: N={len(v):3d}  median={np.median(v):.1f}  '
              f'mean={np.mean(v):.1f}  max={max(v):.1f} km/s')
